# Train a tiny traffic alert model

CPU PyTorch · synthetic data only · no PEMS-BAY download.

Run cells in order. Setup uses a local clone when available; otherwise it downloads this public GitHub repository. PyTorch is installed from its official CPU index only if missing. Local CPU cell execution is verified; hosted Colab has not been executed.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import tempfile

root = Path.cwd()
if not (root / "train_tiny_alert.py").is_file() and (root.parent / "train_tiny_alert.py").is_file():
    root = root.parent
if not (root / "train_tiny_alert.py").is_file():
    root = Path(tempfile.mkdtemp(prefix="traffic-alert-notebook-")) / "repo"
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lanausee-debug/traffic-alert-lab.git", str(root)], check=True)
if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.8.0", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
sys.path.insert(0, str(root))
from train_tiny_alert import make_timeline, run
from alert_eval import evaluate, load_csv


## 1. Inspect the input

Each row has 12 historical speeds followed by 12 availability indicators. Inputs end at t−1; targets look at t..t+2. Labels never enter prediction inputs.


In [ ]:
x, y, _ = make_timeline(11, 2160)
print("Input:", tuple(x.shape), "Target:", tuple(y.shape))
print("First speed window:", x[0, :12].tolist())
print("First observation mask:", x[0, 12:].tolist())


## 2. Actually train

The 24→32→16→1 MLP uses BCEWithLogitsLoss and Adam. Every update runs zero_grad(), backward(), step(). The script verifies parameter changes. Independent timeline seeds separate training (11), calibration (22) and evaluation (33). Threshold selection uses calibration pointwise F1 only.


In [ ]:
import uuid
output = root / "runs" / ("tiny-notebook-" + uuid.uuid4().hex[:8])
record = run(output, steps=120)
print("Optimizer updates:", record["optimizer_updates"])
print("Parameters changed:", record["parameters_changed"])
print("First / last loss:", record["loss_history"][0], record["loss_history"][-1])
print("Frozen threshold:", record["threshold_selected_without_evaluation_labels"])


## 3. Evaluate events, not only loss

Read detection, early recall and false alarms together. Missing scores, censored events and duplicate alerts have explicit rules in docs/csv_evaluation.md.


In [ ]:
scores = load_csv(output / "predictions.csv")
result = evaluate(scores, threshold=record["threshold_selected_without_evaluation_labels"])
result["metrics"]


## 4. Open or download the report

These simulation results are teaching values. They do not replace the frozen thesis results or demonstrate real congestion performance. No weights are saved.


In [ ]:
if importlib.util.find_spec("IPython") is not None:
    from IPython.display import HTML, display
    display(HTML((output / "report.html").read_text(encoding="utf-8")))
else:
    print("Open the HTML report:", output / "report.html")
print("Artifacts:", output)
